In [ ]:
# Figure A.10
#
# Worst-case scaling of the twin count with n: for each n, sample at the
# hardenst density and plot the largest twin count and the largest number of
# isomorphism classes seen across the samples.

In [ ]:
import math
import pathlib

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
import sys
sys.path.append("../python/")
from distance_stats import data_exists, read_data

In [ ]:
sup_title_size = 20
label_size = 18
tick_size = 14
legend_font = 16
labelpad = 3
mpl.rcParams["figure.titlesize"] = sup_title_size
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size
plt.rcParams["lines.markersize"] = 12

In [ ]:
def hardest_m(n, k):
    """The m that maximises the number of distinct k-uniform hypergraphs.

    There are C(n,k) possible hyperedges and we sample m of them, so the number
    of hypergraphs to choose from is C(C(n,k)-1, m). That is largest at
    m = (C(n,k)-1)/2 - the densest point of the search.
    """
    top = math.comb(n, k) - 1
    return max(((h, math.comb(top, h)) for h in range(1, top + 1)),
               key=lambda kv: kv[1])[0]

for k in (3, 4):
    for n in range(6, 10):
        print(f"k={k} n={n}: C(n,k)={math.comb(n, k)}, hardest m={hardest_m(n, k)}")

In [ ]:
results_base = pathlib.Path("../../results/reproduce/")
sampled_results_path = results_base / "density-500/"

num_samples = 500
# The largest n for which the hardest-density cell was actually run.
max_ns = {3: 9, 4: 8}

In [ ]:
fig, axs = plt.subplots(1, 1, squeeze=False, constrained_layout=True)
row = col = 0

for k in (3, 4):
    ns, max_isos_m, max_twins_m = [], [], []
    for n in range(6, max_ns[k] + 1):
        m = hardest_m(n, k)
        filename = sampled_results_path / f"n-{n}_m-{m}_k-{k}_samples-{num_samples}.csv"
        if not data_exists(filename):
            print(f"missing, skipping: {filename}")
            continue

        try:
            output_data = read_data(filename,
                                    xgi_hypergraphs=False)
        except Exception:
            # Report which file failed, then let it propagate. Swallowing this
            # would leave output_data holding the PREVIOUS n's results and
            # plot them under this n.
            print(f"failed to read {filename}")
            raise

        ns.append(n)
        max_isos_m.append(np.array(output_data["total_filtered"]).max())
        max_twins_m.append(np.array(output_data["total_twins"]).max())

    axs[row][col].plot(ns, max_twins_m, label=rf"$k={k}, Twins$", ls="--", c=f"C{k-3}")
    axs[row][col].plot(ns, max_isos_m, label=rf"$k={k}, Iso$", c=f"C{k-3}")

axs[row][col].set_yscale("log")
axs[row][col].set_ylabel("Max Iso. Classes")
axs[row][col].set_xlabel(r"$n$")
axs[row][col].set_xticks(list(range(6, max(max_ns.values()) + 1)))
axs[row][col].legend(frameon=False)
axs[row][col].spines[["top", "right"]].set_visible(False)
fig.savefig("../plots/max-twins-scaling.pdf", bbox_inches="tight")